**Diffusion Models**

In [1]:
from datasets import load_dataset
from util import transforms

dataset = load_dataset(r"D:\DeepLearning\DataSet\Figure\hotdog", split="train", )
dataset.set_transform(transforms)

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

grid = make_grid(dataset["train"][:16]["input"], 8, 2)
plt.figure(figsize=(8,2),dpi=300)
plt.imshow(grid.numpy().transpose((1,2,0)))
plt.axis("off")
plt.show()

Resolving data files:   0%|          | 0/2000 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/800 [00:00<?, ?it/s]

ValueError: Column 'train' doesn't exist.

In [ ]:
import torch

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

resolution=64
batch_size=4
train_dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, shuffle=True)

forward diffusion process

In [ ]:
clean_images=next(iter(train_dataloader))["input"]*2-1
print(clean_images.shape)
nums=clean_images.shape[0]
noise=torch.randn(clean_images.shape)
print(noise.shape)

In [ ]:
from utils import DDIMScheduler

noise_scheduler=DDIMScheduler(num_train_timesteps=1000)
allimgs=clean_images
for step in range(200,1001,200):
    timesteps=torch.tensor([step-1]*4).long()
    noisy_images=noise_scheduler.add_noise(clean_images,
                 noise, timesteps)
    allimgs=torch.cat((allimgs,noisy_images))

import torchvision
imgs=torchvision.utils.make_grid(allimgs,4,6)
fig = plt.figure(dpi=300)
plt.imshow((imgs.permute(2,1,0)+1)/2)
plt.axis("off")
plt.show()

# 3	Build a denoising U-Net model
## 3.1	The attention mechanism in the denoising U-Net model 

In [ ]:
# the Attention() class is defined in ch15util.py
import torch
from torch import nn, einsum
from einops import rearrange

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.scale = dim_head**-0.5
        self.heads = heads
        hidden_dim = dim_head * heads
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias=False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)
    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=1)    #A
        q, k, v = map(
        lambda t: rearrange(t, 'b (h c) x y -> b h c (x y)', h=self.heads),
        qkv)    #B
        q = q * self.scale    
        sim = einsum('b h d i, b h d j -> b h i j', q, k)
        attn = sim.softmax(dim=-1)    #C
        out = einsum('b h i j, b h d j -> b h i d', attn, v)    #D
        out = rearrange(out, 'b h (x y) d -> b (h d) x y', x=h, y=w)
        return self.to_out(out)    #E
attn=Attention(128)
x=torch.rand(1,128,64,64)
out=attn(x)
print(out.shape)

## 3.2	The denoising U-Net model

In [ ]:
from utils.unet_util import UNet

device="cuda" if torch.cuda.is_available() else "cpu"
resolution=64
model=UNet(3,hidden_dims=[128,256,512,1024],
           image_size=resolution).to(device)
num=sum(p.numel() for p in model.parameters())
print("number of parameters: %.2fM" % (num/1e6,))
print(model)

# 4	Train and use the denoising U-Net model
## 4.1 Train the denoising U-Net model

In [ ]:
from diffusers.optimization import get_scheduler

num_epochs=100
optimizer=torch.optim.AdamW(model.parameters(),lr=0.0001,
    betas=(0.95,0.999),weight_decay=0.00001,eps=1e-8)
lr_scheduler=get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=300,
    num_training_steps=(len(train_dataloader) * num_epochs))

In [ ]:
for epoch in range(num_epochs):
    model.train()
    tloss = 0
    print(f"start epoch {epoch}")
    for step, batch in enumerate(train_dataloader):
        clean_images = batch["input"].to(device)*2-1
        nums = clean_images.shape[0]
        noise = torch.randn(clean_images.shape).to(device)
        timesteps = torch.randint(0,
                noise_scheduler.num_train_timesteps,
                (nums, ),
                device=device).long()
        noisy_images = noise_scheduler.add_noise(clean_images,
                     noise, timesteps)

        noise_pred = model(noisy_images, timesteps)["sample"]
        loss = torch.nn.functional.l1_loss(noise_pred, noise)
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        tloss += loss.detach().item()
        if step%100==0:
            print(f"step {step}, average loss {tloss/(step+1)}")

torch.save(model.state_dict(),'files/diffusion.pth')

## 4.2 Use the trained model to generate flower images

```python
# this is defined in the DDIMScheduler() class in ch15util.py
    @torch.no_grad()
    def generate(self,model,device,batch_size=1,generator=None,
         eta=1.0,use_clipped_model_output=True,num_inference_steps=50):
        imgs=[]
        image=torch.randn((batch_size,model.in_channels,model.sample_size,
            model.sample_size),generator=generator).to(device)
        self.set_timesteps(num_inference_steps)
        for t in tqdm(self.timesteps):
            model_output = model(image, t)["sample"]
            image = self.step(model_output,t,image,eta,
                  use_clipped_model_output=use_clipped_model_output)
            img = unnormalize_to_zero_to_one(image)
            img = img.cpu().permute(0, 2, 3, 1).numpy()
            imgs.append(img)
        image = unnormalize_to_zero_to_one(image)
        image = image.cpu().permute(0, 2, 3, 1).numpy()
        return {"sample": image}, imgs
```

In [ ]:
sd=torch.load('files/diffusion.pth')
model.load_state_dict(sd)
with torch.no_grad():
    generator = torch.manual_seed(1)
    generated_images,imgs = noise_scheduler.generate(
        model,device,
        num_inference_steps=50,
        generator=generator,
        eta=1.0,
        use_clipped_model_output=True,
        batch_size=10)
imgnp=generated_images["sample"]    
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4),dpi=300)
for i in range(10):
    ax = plt.subplot(2,5, i + 1)
    plt.imshow(imgnp[i])
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
plt.show()

In [ ]:
# exercise 15.1
with torch.no_grad():
    generator = torch.manual_seed(2)
    generated_images,_ = noise_scheduler.generate(
        model,device,
        num_inference_steps=50,
        generator=generator,
        eta=1.0,
        use_clipped_model_output=True,
        batch_size=10)
imgnp=generated_images["sample"]    
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4),dpi=300)
for i in range(10):
    ax = plt.subplot(2,5, i + 1)
    plt.imshow(imgnp[i])
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
plt.show()

<img src="https://gattonweb.uky.edu/faculty/lium/gai/seed2.png" />

In [ ]:
# keep time steps 800, 600, 400, 200, and 0
steps=imgs[9::10]
# select four sets of flowers out of ten
imgs20=[]
for j in [1,3,6,9]:
    for i in range(5):
        imgs20.append(steps[i][j])
# plot the 20 images in a 4 by 5 grid
plt.figure(figsize=(10,8),dpi=300)
for i in range(20):
    k=i%5
    ax = plt.subplot(4,5, i + 1)
    plt.imshow(imgs20[i])
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.title(f't={800-200*k}',fontsize=15,c="r")
plt.show()

In [ ]:
# https://platform.openai.com/docs/models for a list of models
from openai import OpenAI

openai_api_key=your actual OpenAI API key here, in quotes
client=OpenAI(api_key=openai_api_key)

# generte image
response = client.images.generate(
  model="dall-e-2",
  prompt="an astronaut in a space suit riding a unicorn",
  size="512x512",
  quality="standard",
  n=1,
)
image_url = response.data[0].url
print(image_url)

<img src="https://gattonweb.uky.edu/faculty/lium/gai/unicorn.png" />

In [ ]:
# exercise 15.2
response = client.images.generate(
  model="dall-e-2",
  prompt="a cat in a suit working on a computer",
  size="512x512",
  quality="standard",
  n=1,
)    
image_url = response.data[0].url
print(image_url)

<img src="https://gattonweb.uky.edu/faculty/lium/gai/catsuit.png" />